In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["PATH"] = "/mnt/lustre-grete/usr/u12045/projects/LLAVA-Med/envs/lerobot/bin:" + os.environ.get("PATH", "")
os.environ["HF_HOME"] = "/mnt/lustre-grete/usr/u12045/vla/hf_cache"
os.environ["TMPDIR"] = "/mnt/lustre-grete/usr/u12045/vla/cache"
os.environ["PYTHONPATH"] = "/mnt/lustre-grete/usr/u12045/vla/duci/VLA-Humanoid:" + os.environ.get("PYTHONPATH", "")
import sys
sys.path.insert(0, "/mnt/lustre-grete/usr/u12045/vla/duci/VLA-Humanoid")


In [2]:

import argparse
from collections import Counter, defaultdict
import logging
import os
from pathlib import Path
import sys
import time
from moviepy.editor import ImageSequenceClip
import copy
import json

from typing import Optional, Sequence, List

import matplotlib.pyplot as plt
import numpy as np
from transformers import AutoModel, AutoProcessor
from collections import deque
from PIL import Image
import torch
import cv2 as cv

# This is for using the locally installed repo clone when using slurm
from calvin.calvin_models.calvin_agent.models.calvin_base_model import CalvinBaseModel

# sys.path.insert(0, Path(__file__).absolute().parents[2].as_posix())

from calvin.calvin_models.calvin_agent.evaluation.multistep_sequences import get_sequences
from calvin.calvin_models.calvin_agent.evaluation.utils import (
    collect_plan,
    count_success,
    create_tsne,
    get_default_model_and_env,
    get_env_state_for_initial_condition,
    get_log_dir,
    join_vis_lang,
    print_and_save,
)
from calvin.calvin_models.calvin_agent.utils.utils import get_all_checkpoints, get_checkpoints_for_epochs, get_last_checkpoint
import hydra
import numpy as np
from omegaconf import OmegaConf
from pytorch_lightning import seed_everything
from termcolor import colored
import torch
from tqdm.auto import tqdm
import numpy as np
from calvin_env.envs.play_table_env import get_env

logger = logging.getLogger(__name__)

%load_ext autoreload
%autoreload 2

error: XDG_RUNTIME_DIR is invalid or not set in the environment.
Failed to create random directory /mnt/lustre-grete/usr/u12045/vla/cache/pulse-8n2L9K88eMlG: No such file or directory
ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5205:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5205:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1342:(snd_func_refer) error evaluating name
ALSA lib conf.c:5205:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5728:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2722:(snd_pcm_open_noupdate) Unknown PCM default
Failed to create random directory /mnt/lustre-grete/usr/u12045/vla/cache/pulse-n7CcctT9RwKS: No such file or directory
ALSA lib confmisc.c:855:(p

In [3]:
def get_epoch(checkpoint):
    if "=" not in checkpoint.stem:
        return "0"
    checkpoint.stem.split("=")[1]

def make_env(dataset_path):
    val_folder = Path(dataset_path) / "validation"
    env = get_env(val_folder, show_gui=False)

    # insert your own env wrapper
    # env = Wrapper(env)
    return env
np.float = float
# env = make_env("/my_data/CALVIN/all_scale_D/scale_100")
# env = make_env("/mnt/vast-kisski/projects/kisski-umg-fairpact-2/VLA/CALVIN/all_scale_D/scale_100")
env = make_env("/mnt/lustre-grete/usr/u12045/vla/hf_cache/h5_calvin/all_scale_D/scale_100")


argv[0]=--width=200
argv[1]=--height=200
Loaded EGL 1.5 after reload.
GL_VENDOR=NVIDIA Corporation
GL_RENDERER=NVIDIA A100-SXM4-80GB/PCIe/SSE2
GL_VERSION=3.3.0 NVIDIA 550.163.01
GL_SHADING_LANGUAGE_VERSION=3.30 NVIDIA via Cg compiler
Version = 3.3.0 NVIDIA 550.163.01
Vendor = NVIDIA Corporation
Renderer = NVIDIA A100-SXM4-80GB/PCIe/SSE2


EGL device choice: -1 of 6.


received depth0
WARNING uncommitted modified files: lerobot/common/policies/pi0/modeling_pi0.py,lerobot/common/policies/pi0/paligemma_with_expert.py,myutils/pi0_infer.py,notebooks/calvin_evaluation.ipynb,notebooks/visualize_attn_map_train_data.ipynb


In [4]:

def evaluate_sequence(env, model, task_checker, initial_state, eval_sequence, val_annotations, plans, debug, eval_log_dir):
    """
    Evaluates a sequence of language instructions.
    """
    robot_obs, scene_obs = get_env_state_for_initial_condition(initial_state)
    env.reset(robot_obs=robot_obs, scene_obs=scene_obs)

    success_counter = 0
    if debug:
        time.sleep(1)
        print()
        print()
        print(f"Evaluating sequence: {' -> '.join(eval_sequence)}")
        print("Subtask: ", end="")
    for subtask in eval_sequence:
        success = rollout(env, model, task_checker, subtask, val_annotations, plans, debug, eval_log_dir)
        if success:
            success_counter += 1
        else:
            return success_counter
    return success_counter

In [5]:
import sys
sys.argv = ['']

EP_LEN = 360
NUM_SEQUENCES = 3
def evaluate_policy(model, env, epoch=-1, eval_log_dir=None, debug=False, create_plan_tsne=False):
    """
    Run this function to evaluate a model on the CALVIN challenge.

    Args:
        model: Must implement methods of CalvinBaseModel.
        env: (Wrapped) calvin env.
        epoch:
        eval_log_dir: Path where to log evaluation results. If None, logs to /tmp/evaluation/
        debug: If True, show camera view and debug info.
        create_plan_tsne: Collect data for TSNE plots of latent plans (does not work for your custom model)

    Returns:
        Dictionary with results
    """
    conf_dir = Path("../calvin/calvin_models") / "conf"
    task_cfg = OmegaConf.load(conf_dir / "callbacks/rollout/tasks/new_playtable_tasks.yaml")
    task_oracle = hydra.utils.instantiate(task_cfg)
    val_annotations = OmegaConf.load(conf_dir / "annotations/new_playtable_validation.yaml")
    eval_log_dir = get_log_dir(eval_log_dir)

    eval_sequences = get_sequences(NUM_SEQUENCES)
    with open("../calvin/eval_sequences.json", "r") as f:
        eval_sequences = json.load(f)
    eval_sequences = eval_sequences[0:3]


    results = []
    plans = defaultdict(list)

    if not debug:
        eval_sequences = tqdm(eval_sequences, position=0, leave=True)

    for initial_state, eval_sequence in eval_sequences:
        result = evaluate_sequence(env, model, task_oracle, initial_state, eval_sequence, val_annotations, plans, debug, eval_log_dir)
        results.append(result)
        if not debug:
            eval_sequences.set_description(
                " ".join([f"{i + 1}/5 : {v * 100:.1f}% |" for i, v in enumerate(count_success(results))]) + "|"
            )

    if create_plan_tsne:
        create_tsne(plans, eval_log_dir, epoch)
    print_and_save(results, eval_sequences, eval_log_dir, epoch)

    return results
    
def main(env, model):
    # seed_everything(0, workers=True)  # type:ignore
    parser = argparse.ArgumentParser(description="Evaluate a trained model on multistep sequences with language goals.")
    parser.add_argument("--dataset_path", type=str, help="Path to the dataset root directory.", default= "/my_data/CALVIN/all_scale_D/scale_10")

    # arguments for loading default model
    parser.add_argument(
        "--train_folder", type=str, help="If calvin_agent was used to train, specify path to the log dir."
    )
    parser.add_argument(
        "--checkpoints",
        type=str,
        default=None,
        help="Comma separated list of epochs for which checkpoints will be loaded",
    )
    parser.add_argument(
        "--checkpoint",
        type=str,
        default=None,
        help="Path of the checkpoint",
    )
    parser.add_argument(
        "--last_k_checkpoints",
        type=int,
        help="Specify the number of checkpoints you want to evaluate (starting from last). Only used for calvin_agent.",
    )

    # arguments for loading custom model or custom language embeddings
    parser.add_argument(
        "--custom_model", action="store_true", help="Use this option to evaluate a custom model architecture."
    )

    parser.add_argument("--debug", action="store_true", help="Print debug info and visualize environment.", default= True)

    parser.add_argument("--eval_log_dir", type=str, help="Where to log the evaluation results.", default= "../calvin_eval_logs_debug")

    parser.add_argument("--device", default=0, type=int, help="CUDA device")
    args = parser.parse_args()

    env.reset()
    
    evaluate_policy(model, env, debug=args.debug, eval_log_dir=args.eval_log_dir)

In [6]:
# evaluate a custom model
from myutils.pi0_infer import Pi0TorchInference, Pi0JaxInference

ckpt_jax_dir = Path(f"/mnt/lustre-grete/usr/u12045/vla/duci/openpi/checkpoints/pi0_calvin_50%_joint/pi0_calvin_50%_joint/30000")
dataset_repo_id = "ducido/calvin_task_D_D_scale_50_lerobo_format"
with open(ckpt_jax_dir / f"assets/{dataset_repo_id}/norm_stats.json") as f:
    norm_stats = json.load(f)

model_path = "/mnt/lustre-grete/usr/u12045/vla/duci/VLA-Humanoid/calvin_jaxcp_conversion_to_torch"
model = Pi0JaxInference(
    model_dir=model_path,
    dataset_repo_id=dataset_repo_id,
    norm_stats=norm_stats,
    device='cuda'
)


# model_path = '../outputs/train/2025-07-26/11-20-33_calvin_100%_defaultconfig_removepaddingloss/checkpoints/100000/pretrained_model'
# model = Pi0TorchInference(
#     model_dir=model_path,
#     device='cuda'
# )

The dataset you requested (ducido/calvin_task_D_D_scale_50_lerobo_format) is in 2.0 format.
While current version of LeRobot is backward-compatible with it, the version of your dataset still uses global
stats instead of per-episode stats. Update your dataset stats to the new format using this command:
```
python lerobot/common/datasets/v21/convert_dataset_v20_to_v21.py --repo-id=ducido/calvin_task_D_D_scale_50_lerobo_format
```

If you encounter a problem, contact LeRobot maintainers on [Discord](https://discord.com/invite/s3KuuzsPFb)
or open an [issue on GitHub](https://github.com/huggingface/lerobot/issues/new/choose).



load pretrained policy
Loading weights from local directory


In [7]:
rm -r ../calvin_eval_logs_debug/*

rm: cannot remove '../calvin_eval_logs_debug/*': No such file or directory


In [8]:
import time
store_time = []
i = 0
def rollout(env, model, task_oracle, subtask, val_annotations, plans, debug, eval_log_dir):
    global i
    """
    Run the actual rollout on one subtask (which is one natural language instruction).
    """
    if debug:
        print(f"{subtask} ", end="")
        time.sleep(0.5)
        img_list = []
    obs = env.get_obs()


    
    # get lang annotation for subtask
    print(subtask)
    lang_annotation = val_annotations[subtask][0]


    start_info = env.get_info()
    prefix_gif = f"{i}_jax"
    i += 1
    for step in range(0, EP_LEN//5):
        ### Reformat obs
        obs = {
            'image': obs["rgb_obs"]["rgb_static"],
            'wrist_image':obs["rgb_obs"]["rgb_gripper"],
            "state": obs["robot_obs"],
            "task": str(lang_annotation),
        }
        action_chunk = model.calvin_step(obs)
        action_chunk[:, -1] = np.where(action_chunk[:, -1] > 0, 1, -1)
        # print(action_chunk.shape)
        # model.policy._action_queue = deque()  # reset action queue
        # visualize_attention(attentions, image_rgb, lang_annotation)
        # print(action.shape)
        for action in action_chunk[:10]:
            action = {"action": action, "type": "joint_abs"}
            obs, _, _, current_info = env.step(action)

            if debug:
                # img = env.render(mode="rgb_array")
                # join_vis_lang(img, lang_annotation)
                img_copy = copy.deepcopy(obs["rgb_obs"]["rgb_static"])
                img_list.append(img_copy)
                # time.sleep(0.1)
            if step == 0:
                # for tsne plot, only if available
                collect_plan(model, plans, subtask)

            # check if current step solves a task
            current_task_info = task_oracle.get_task_info_for_set(start_info, current_info, {subtask})
            if len(current_task_info) > 0:
                if debug:
                    print(colored("success", "green"), end=" ")
                    clip = ImageSequenceClip(img_list, fps=30)
                    clip.write_gif(
                        os.path.join(
                            eval_log_dir, f"{prefix_gif}_{subtask}-succ.gif"
                        ),
                        fps=30,
                    )
                return True

    if debug:
        print(colored("fail", "red"), end=" ")
        clip = ImageSequenceClip(img_list, fps=30)
        clip.write_gif(
            os.path.join(eval_log_dir, f"{prefix_gif}_{subtask}-fail.gif"),
            fps=30,
        )
    return False

main(env, model)

ven = NVIDIA Corporation
ven = NVIDIA Corporation
logging to ../calvin_eval_logs_debug


Evaluating sequence: rotate_blue_block_right -> move_slider_right -> lift_red_block_slider -> place_in_slider -> turn_off_lightbulb
Subtask: rotate_blue_block_right rotate_blue_block_right
fail MoviePy - Building file ../calvin_eval_logs_debug/0_jax_rotate_blue_block_right-fail.gif with imageio.




Evaluating sequence: turn_off_led -> push_into_drawer -> lift_blue_block_drawer -> place_in_slider -> close_drawer
Subtask: turn_off_led turn_off_led
success MoviePy - Building file ../calvin_eval_logs_debug/1_jax_turn_off_led-succ.gif with imageio.


push_into_drawer push_into_drawer
success MoviePy - Building file ../calvin_eval_logs_debug/2_jax_push_into_drawer-succ.gif with imageio.


lift_blue_block_drawer lift_blue_block_drawer
success MoviePy - Building file ../calvin_eval_logs_debug/3_jax_lift_blue_block_drawer-succ.gif with imageio.


place_in_slider place_in_slider
success MoviePy - Building file ../calvin_eval_logs_debug/4_jax_place_in_slider-succ.gif with imageio.


...

place_in_slider place_in_slider
success MoviePy - Building file ../calvin_eval_logs_debug/7_jax_place_in_slider-succ.gif with imageio.


open_drawer 

open_drawer
success MoviePy - Building file ../calvin_eval_logs_debug/8_jax_open_drawer-succ.gif with imageio.


rotate_red_block_right rotate_red_block_right
fail MoviePy - Building file ../calvin_eval_logs_debug/9_jax_rotate_red_block_right-fail.gif with imageio.


Results for Epoch -1:
Average successful sequence length: 2.6666666666666665
Success rates for i instructions in a row:
1: 66.7%
2: 66.7%
3: 66.7%
4: 33.3%
5: 33.3%
turn_off_led: 1 / 1 |  SR: 100.0%
push_into_drawer: 1 / 1 |  SR: 100.0%
lift_blue_block_drawer: 1 / 1 |  SR: 100.0%
place_in_slider: 2 / 2 |  SR: 100.0%
close_drawer: 1 / 1 |  SR: 100.0%
lift_pink_block_slider: 1 / 1 |  SR: 100.0%
open_drawer: 1 / 1 |  SR: 100.0%
rotate_blue_block_right: 0 / 1 |  SR: 0.0%
rotate_red_block_right: 0 / 1 |  SR: 0.0%

Best model: epoch -1 with average sequences length of 2.6666666666666665


state torch.Size([1, 15]) torch.float32

image torch.Size([1, 3, 200, 200]) torch.float32

wrist_image torch.Size([1, 3, 84, 84]) torch.float32

In [9]:
obs

NameError: name 'obs' is not defined